# Gemación — 10.000 embeddings reales en T4

Cierra el límite de R11: la prueba a escala usaba un generador calibrado (media + 100 componentes
+ residuo isotrópico). Acá se usan embeddings **reales** a escala.

**Modelo:** `gemma:2b` vía Ollama, Q4_0 — exactamente el mismo blob de pesos que `albert:v4.0`
en la PC (verificado: `sha256-c1864a5e…` en ambos, sin `ADAPTER`). Se usa Ollama en vez de
HuggingFace para replicar la misma cuantización y el mismo pooling que los 800 vectores locales,
y de paso evitar el gating de Gemma en HF.

**Runtime:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.

Salida: `embeddings_10k.npy` (10000 × 2048, float32) + el análisis impreso.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull gemma:2b
!ollama list

In [ ]:
# Corpus: 10.000 párrafos naturales y diversos (wikitext-103).
# Importa que sean textos reales: la pregunta es la geometría del espacio de embeddings,
# y un corpus sintético repetitivo la falsearía hacia menos diversidad de la real.
!pip -q install datasets
from datasets import load_dataset

ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
textos, vistos = [], set()
for r in ds:
    t = r['text'].strip()
    if 120 < len(t) < 1000 and not t.startswith('=') and t[:60] not in vistos:
        vistos.add(t[:60]); textos.append(t)
    if len(textos) >= 10000:
        break
print(len(textos), 'textos |', textos[0][:120])

In [ ]:
# Embeddings. Ollama no batchea, así que se paraleliza con hilos (la GPU multiplexa).
import numpy as np, json, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

def emb(t):
    d = json.dumps({'model': 'gemma:2b', 'prompt': t}).encode()
    for _ in range(3):
        try:
            r = urllib.request.urlopen(urllib.request.Request(
                'http://localhost:11434/api/embeddings', data=d,
                headers={'Content-Type': 'application/json'}), timeout=120)
            return json.load(r)['embedding']
        except Exception:
            time.sleep(2)
    return None

vecs, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    for i, v in enumerate(ex.map(emb, textos)):
        if v is not None:
            vecs.append(v)
        if (i + 1) % 1000 == 0:
            print(f'{i+1}/{len(textos)} — {time.time()-t0:.0f}s')

X = np.array(vecs, dtype=np.float32)
X = X / np.linalg.norm(X, axis=1, keepdims=True)
np.save('embeddings_10k.npy', X)
print('listo:', X.shape, f'{time.time()-t0:.0f}s')

In [ ]:
# --- Análisis: perfil del espacio (replica R11.1 con n >> d) ---
import numpy as np
X = np.load('embeddings_10k.npy'); n, d = X.shape
rng = np.random.default_rng(0)

def perfil(Y, nombre):
    m, dd = Y.shape
    i, j = rng.integers(0, m, 50000), rng.integers(0, m, 50000); k = i != j
    cos = np.sum(Y[i[k]] * Y[j[k]], 1)
    C = np.cov(Y.T) if dd <= m else None
    lam = np.clip(np.linalg.eigvalsh(C), 0, None) if C is not None else None
    pr = (lam.sum()**2 / np.sum(lam**2)) if lam is not None else float('nan')
    print(f'{nombre:>28} d={dd:5d} |cos|={np.mean(np.abs(cos)):.4f} '
          f'sd={np.std(cos):.4f} dim_efectiva={pr:8.1f}')

def esfera(r, m, dd):
    x = r.normal(size=(m, dd)).astype(np.float32)
    return x / np.linalg.norm(x, axis=1, keepdims=True)

print('n =', n, '> d =', d, '→ la covarianza ahora SÍ se estima bien (en R11 era n=800 < d)')
perfil(X, 'gemma:2b real')
perfil(X - X.mean(0), 'gemma:2b centrado')
perfil(esfera(rng, n, d), f'uniforme S^{d-1}')
print('norma del vector medio:', np.linalg.norm(X.mean(0)).round(4))

In [ ]:
# --- R3 a escala con embeddings REALES (sin generador) ---
import numpy as np
X = np.load('embeddings_10k.npy')

def esf(r, m, dd):
    x = r.normal(size=(m, dd)).astype(np.float32)
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def tangente(t, x):
    t = t - (t * x).sum(-1, keepdims=True) * x
    return t / (np.linalg.norm(t, axis=-1, keepdims=True) + 1e-8)

def r3(seed, base, K=4, eps=0.3, alpha=0.4, delta=3.0, ruido=0.05, Q=300):
    rng = np.random.default_rng(seed); N, d = base.shape
    that = esf(rng, N, d); cur = base.copy(); vs = [base.copy()]
    for _ in range(K):
        u = alpha * tangente(that, cur) + (1 - alpha) * esf(rng, N, d)
        cur = cur + eps * tangente(u, cur)
        cur /= np.linalg.norm(cur, axis=1, keepdims=True); vs.append(cur.copy())
    A = np.concatenate(vs, 0).astype(np.float32)
    mem = np.tile(np.arange(N), K + 1); ver = np.repeat(np.arange(K + 1), N)
    qi = rng.choice(N, min(Q, N), replace=False)
    q0 = base[qi] + ruido * esf(rng, len(qi), d)
    q0 /= np.linalg.norm(q0, axis=1, keepdims=True)
    h1 = np.argmax(q0 @ A.T, 1)
    q = q0 + delta * tangente(that[mem[h1]], q0)
    q /= np.linalg.norm(q, axis=1, keepdims=True)
    t1 = np.argmax(q @ A.T, 1); ok = mem[t1] == qi
    return float(np.mean(ok & (ver[t1] == K))), float(np.mean(ok))

print(f"{'base':>26} {'N':>7} {'M1':>8} {'M2':>8}")
rng = np.random.default_rng(1)
for N in (1000, 5000, 10000):
    if N > len(X):
        break
    sub = X[rng.choice(len(X), N, replace=False)]
    r = [r3(500 + s, sub) for s in range(3)]
    print(f"{'gemma real':>26} {N:7d} {np.mean([a for a,_ in r]):8.3f} "
          f"{np.mean([b for _,b in r]):8.3f}")
    r = [r3(500 + s, esf(np.random.default_rng(600+s), N, 16)) for s in range(3)]
    print(f"{'uniforme d=16 (control)':>26} {N:7d} {np.mean([a for a,_ in r]):8.3f} "
          f"{np.mean([b for _,b in r]):8.3f}")

In [ ]:
from google.colab import files
files.download('embeddings_10k.npy')   # traerlo a la PC para cruzarlo con el resto